# Frozen lake: Q-learning vs SARSA

A port of `frozenlakeQlearningSarsa.py` to a notebook.

The frozen lake is a $4\times4$ grid from OpenAI Gymnasium. The player starts
at the top-left and must reach the goal at the bottom-right without falling
into a hole. The lake is **slippery**: the commanded move happens with
probability $1/3$, and each of the two perpendicular moves with probability
$1/3$. A reward of $1$ is given for reaching the goal, and $0$ otherwise.

Two algorithms are compared:

- **Q-learning** (Unit 10), off-policy: the target is
  $g+\gamma\max_v\hat Q(x_{k+1},v)$, the best action at the next state,
  regardless of what the exploration policy actually does there.
- **SARSA** (Unit 11), on-policy: the target is
  $g+\gamma\hat Q(x_{k+1},u_{k+1})$, using the action the $\epsilon$-greedy
  policy actually took.

One line of code separates them, and this notebook is about what that line
does to the result.

::: note
A convention warning. The rest of the course **minimises a cost**; Gymnasium
hands out **rewards to maximise**, so here the greedy action is an
$\arg\max$, not an $\arg\min$. The algorithms are identical up to that sign.
:::

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation

try:
    import gymnasium as gym
    HAVE_GYM = True
except ImportError:
    HAVE_GYM = False
print("gymnasium available:", HAVE_GYM)

In [ ]:
# A 25-line stand-in for gymnasium's FrozenLake-v1, so the notebook runs
# with nothing installed. If gymnasium is present it is used instead, and the
# two give the same dynamics.
MAP4 = ["SFFF", "FHFH", "FFFH", "HFFG"]

class MiniFrozenLake:
    """Slippery 4x4 frozen lake. Actions: 0 left, 1 down, 2 right, 3 up."""
    def __init__(self, desc=MAP4):
        self.desc = np.array([[c.encode() for c in row] for row in desc])
        self.n = len(desc)
        self.nS, self.nA = self.n*self.n, 4
        self.unwrapped = self
        self.observation_space = type("S", (), {"n": self.nS})()
        self.action_space = type("A", (), {"n": self.nA})()
    def reset(self, **kw):
        self.s = 0
        return self.s, {}
    def step(self, a):
        r, c = divmod(self.s, self.n)
        a = random.choice([(a-1) % 4, a, (a+1) % 4])       # slip
        dr, dc = {0:(0,-1), 1:(1,0), 2:(0,1), 3:(-1,0)}[a]
        r2 = min(max(r+dr, 0), self.n-1); c2 = min(max(c+dc, 0), self.n-1)
        self.s = r2*self.n + c2
        ch = self.desc[r2][c2].decode()
        return self.s, (1.0 if ch == "G" else 0.0), ch in "GH", False, {}
    def close(self): pass

def make_env(env_id="FrozenLake-v1", is_slippery=True):
    if HAVE_GYM:
        return gym.make(env_id, is_slippery=is_slippery)
    return MiniFrozenLake()

def reset_env(env):
    res = env.reset()
    return res[0] if isinstance(res, tuple) else res

def step_env(env, action):
    res = env.step(action)
    if len(res) == 5:
        obs, reward, terminated, truncated, info = res
        return obs, reward, (terminated or truncated), info
    obs, reward, done, info = res
    return obs, reward, done, info

In [ ]:
def epsilon_greedy(Q, state, nA, eps):
    if random.random() < eps:
        return random.randrange(nA)
    return int(np.argmax(Q[state]))


def run_q_learning(env, num_episodes=5000, alpha=0.1, gamma=0.99,
                   eps_start=0.5, eps_end=0.05, eps_decay=0.9995):
    nS, nA = env.observation_space.n, env.action_space.n
    Q = np.zeros((nS, nA)); rewards = []; eps = eps_start
    for ep in range(num_episodes):
        state = reset_env(env); done = False; total_r = 0.0
        while not done:
            action = epsilon_greedy(Q, state, nA, eps)
            next_state, reward, done, _ = step_env(env, action)
            # OFF-POLICY: the target uses the best action at the next state
            best_next = np.max(Q[next_state])
            Q[state, action] += alpha*(reward + gamma*best_next - Q[state, action])
            state = next_state; total_r += reward
        rewards.append(total_r); eps = max(eps_end, eps*eps_decay)
    return Q, rewards


def run_sarsa(env, num_episodes=5000, alpha=0.1, gamma=0.99,
              eps_start=0.5, eps_end=0.05, eps_decay=0.9995):
    nS, nA = env.observation_space.n, env.action_space.n
    Q = np.zeros((nS, nA)); rewards = []; eps = eps_start
    for ep in range(num_episodes):
        state = reset_env(env)
        action = epsilon_greedy(Q, state, nA, eps)
        done = False; total_r = 0.0
        while not done:
            next_state, reward, done, _ = step_env(env, action)
            next_action = epsilon_greedy(Q, next_state, nA, eps)
            # ON-POLICY: the target uses the action actually taken next
            Q[state, action] += alpha*(reward + gamma*Q[next_state, next_action]
                                       - Q[state, action])
            state, action = next_state, next_action
            total_r += reward
        rewards.append(total_r); eps = max(eps_end, eps*eps_decay)
    return Q, rewards


def moving_average(x, w=100):
    return np.convolve(x, np.ones(w)/w, mode="valid")

def policy_from_Q(Q):
    return np.argmax(Q, axis=1)

In [ ]:
random.seed(0); np.random.seed(0)
env = make_env("FrozenLake-v1", is_slippery=True)
episodes = 8000

Q_q,  rew_q     = run_q_learning(env, num_episodes=episodes)
Q_s,  rew_sarsa = run_sarsa(env,      num_episodes=episodes)
print(f"Q-learning: final Q[0].max = {Q_q[0].max():.3f}")
print(f"SARSA     : final Q[0].max = {Q_s[0].max():.3f}")

In [ ]:
window = 200
ma_q  = moving_average(rew_q, window)
ma_s  = moving_average(rew_sarsa, window)
episodes_x = np.arange(len(ma_q)) + window
succ_q = moving_average([1.0 if r > 0 else 0.0 for r in rew_q], window)
succ_s = moving_average([1.0 if r > 0 else 0.0 for r in rew_sarsa], window)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(episodes_x, ma_q, label="Q-learning (avg reward)")
ax.plot(episodes_x, ma_s, label="SARSA (avg reward)")
ax.set_xlabel("Episode"); ax.set_ylabel(f"Moving avg reward (window={window})")
ax.grid(True); ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(episodes_x, succ_q, "--", color="C0", alpha=0.6, label="Q-learning (win rate)")
ax2.plot(episodes_x, succ_s, "--", color="C1", alpha=0.6, label="SARSA (win rate)")
ax2.set_ylabel("Win rate"); ax2.legend(loc="lower right")
ax.set_title("FrozenLake: Q-learning vs SARSA")
plt.tight_layout(); plt.show()

In [ ]:
def show_policy(Q, name, n=4):
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    grid = np.array([arrows[a] for a in policy_from_Q(Q)]).reshape((n, n))
    print(f"\n{name} deterministic policy (state grid):")
    for row in grid:
        print(" ".join(row))

show_policy(Q_q, "Q-learning")
show_policy(Q_s, "SARSA")

Both reach a win rate of roughly $0.5$ on the slippery lake, and the
policies they settle on are similar -- which is the point the lecture makes.

The arrows are worth staring at: several of them point *into* a wall. On a
slippery lake that is correct behaviour. Commanding "left" against the left
wall means the two slips are "up" and "down" and the intended move does
nothing, so it is a way of moving vertically while making it impossible to
slide into the hole on the right. A policy that looks wrong is exploiting the
noise.

In [ ]:
# one episode under a given policy, animated on the grid
def _desc_chars(env):
    desc = env.unwrapped.desc
    return np.array([[c.decode("utf-8") for c in row] for row in desc])

def _grid_image(desc_chars):
    n = desc_chars.shape[0]
    img = np.ones((n, n, 3), dtype=np.float32)
    colour = {"S": (0.8, 1.0, 0.8), "F": (1.0, 1.0, 1.0),
              "H": (0.2, 0.2, 0.2), "G": (1.0, 0.85, 0.2)}
    for i in range(n):
        for j in range(n):
            img[i, j] = colour[desc_chars[i, j]]
    return img

def animate_policy(env, policy_fn, title="Policy", max_steps=50, interval=300):
    desc = _desc_chars(env); n = desc.shape[0]
    states = []; obs = reset_env(env); done = False; steps = 0
    while not done and steps < max_steps:
        states.append(obs)
        obs, _, done, _ = step_env(env, int(policy_fn(obs)))
        steps += 1
    states.append(obs)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(_grid_image(desc), origin="upper")
    scat = ax.scatter([], [], s=300, c="red", edgecolors="black", linewidths=1.5)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)

    def update(frame):
        r, c = divmod(int(states[frame]), n)
        scat.set_offsets([[c, r]])
        return (scat,)

    anim = animation.FuncAnimation(fig, update, frames=len(states),
                                   interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


nA = env.action_space.n
animate_policy(env, lambda s: random.randrange(nA), title="Initial (random) policy")

In [ ]:
pi_final = policy_from_Q(Q_q)
animate_policy(env, lambda s: int(pi_final[int(s)]),
               title="Final Q-learning deterministic policy")

In [ ]:
env.close()